# Study Case 01 — Source Profile

Python is used only to execute/display SQL results. All profiling logic is SQL-based.


In [ ]:
import os
import psycopg
from IPython.display import display
conn = psycopg.connect(os.environ['SUPABASE_DB_URL'])


## Source profile

Validate row count, distinct ABN, duplicate excess, and valid identifiers.


In [ ]:
sql = """
SELECT COUNT(*) raw_rows,
       COUNT(DISTINCT regexp_replace(trim(abn),'[^0-9]','','g')) distinct_abn,
       COUNT(*)-COUNT(DISTINCT regexp_replace(trim(abn),'[^0-9]','','g')) duplicate_excess,
       COUNT(*) FILTER (WHERE regexp_replace(trim(abn),'[^0-9]','','g') ~ '^[0-9]{11}$') valid_abn
FROM staging.acnc_ais_raw;
"""
with conn.cursor() as cur:
    cur.execute(sql); display(cur.fetchall())


## Column completeness

Use PostgreSQL to calculate non-null and NULL counts for all fields.


In [ ]:
sql = """
SELECT e.key AS column_name,
       COUNT(*) AS rows,
       COUNT(*) FILTER (WHERE e.value <> 'null'::jsonb) AS non_nulls,
       COUNT(*) FILTER (WHERE e.value = 'null'::jsonb) AS nulls,
       ROUND(100.0*COUNT(*) FILTER (WHERE e.value='null'::jsonb)/COUNT(*),2) AS null_pct
FROM staging.acnc_ais_raw r
CROSS JOIN LATERAL jsonb_each(to_jsonb(r)) e
WHERE e.key NOT IN ('source_load_id','source_file','etl_batch_id','source_loaded_at')
GROUP BY e.key ORDER BY null_pct DESC, column_name;
"""
with conn.cursor() as cur:
    cur.execute(sql); display(cur.fetchall())
